# 3 · Validity & reward-hacking  `[EVAL]`

**Are the score gains real MI skill or a style shift?** — Level 3 of the drill-down: the cross-cutting validity analyses. The rubric factor structure (one global-evaluation halo axis?), the reward-hacking synthesis + its deterministic cross-checks, session shape (text metrics — no oracle at all), and ground-truth transcripts. Exports → `results/<VIEW>/figures|tables/3_validity/`.

Global outcomes → `1_Outcomes`; the per-questionnaire decomposition this synthesis draws on (MITI behaviours, MICI per-behaviour, PCT patient detail, Q2 reward composition) → `2_Questionnaire_Detail`; persona splits → `4_Heterogeneity`; reward faithfulness → `5_Training_and_Reliability`; stats tables → `7_Stats`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="3_validity",             # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

## 1 · Rubric factor structure — two metric families  `[EVAL]`
**Purpose.** Are the rubrics independent skills or one halo? The battery was *designed* as two families, but the data **revises the boundary** — made explicit below in the correlation heatmap (heavy block divider) and the loadings bars (blue vs orange):

- **Global-evaluation (halo) cluster — one factor (blue).** `Q1+Q2`, `WAI-SR`, `CSQ-8`, `MI-SAT`, `MITI` (globals) — subjective satisfaction / alliance / integrity ratings. An **empirical redundancy set, not one official construct** (Q2 and WAI-SR even overlap by design — both alliance measures; per-instrument provenance: `METRICS_REFERENCE.md` §1). On their own they collapse to **one factor (PC1≈91%)** — the single-oracle halo — so "all global-eval rubrics rise together" reflects a single latent axis, **not** multi-skill gain.
- **Intended orthogonal axes (orange)** — built to load *off* that factor: **`PCT`** (patient change-talk proportion — the MI *goal*), **`MICI ↓`** (MI-inconsistent / sycophancy rate, lower = better), and the *free* objective MITI-proficiency ratios **`R:Q` / `%CR` / `%MICO`** (derived from existing behavior counts, no rescoring).

**Finding — PCT is NOT the clean orthogonal axis it was intended as.** Empirically `PCT` (change-talk proportion) loads **with the halo family** (Spearman ρ≈0.79–0.94; high PC1 loading): warmer, more satisfying sessions also elicit more patient change-talk, so PCT does not isolate MI *technique*. The **genuine second axis** is **`MICI ↓` + the MITI ratios `R:Q` / `%CR` / `%MICO`** (objective reflection-vs-persuasion technique). **Read:** adding the orthogonal axes still drops PC1 from ≈91% to **≈55%** — a real second dimension exists — but it is defined by MI-inconsistency + technique ratios, **not** by PCT. (Reward faithfulness lives in `5_Training_and_Reliability`.)

In [ ]:
# Expand the scores with the FREE objective MITI-proficiency ratios (+ PCT/MICI if already scored).
SCO = eda_analysis.add_derived_mitiprof_rows(S.SCORES, S.ARMS)
EXPANDED = [m for m in (eda_analysis.WARMTH_RUBRICS + eda_analysis.ORTHOGONAL_METRICS)
            if m in SCO.questionnaire.unique()]
print("metrics in correlation/PCA:", EXPANDED)

# Inter-rubric correlation over the EXPANDED set (global-eval halo rubrics + orthogonal axes).
fig = plots.rubric_correlation_heatmap(SCO, metrics=EXPANDED)
eda_analysis.save_fig(fig, "rubric_correlation", caption="Spearman correlation among rubric + orthogonal-axis scores (per conversation, pooled). The global-eval (halo) rubrics block together; PCT loads WITH them (rho~0.79-0.94, not orthogonal as intended), while MICI + the MITI ratios R:Q/%CR/%MICO form the genuine second family."); plt.show()

# PCA: pooled + per arm. Does adding orthogonal axes drop the dominant PC1 share?
pca_all = stats.rubric_pca(SCO, metrics=EXPANDED)
if pca_all["explained_variance_ratio"]:
    print(f"\nPOOLED: PC1 = {pca_all['explained_variance_ratio'][0]:.1%}  "
          f"(over {len(pca_all['metrics'])} metrics: {pca_all['metrics']})")
    print(f"        PC1 loadings = {pca_all['pc1_loadings']}")
    print("        -> global-eval (halo) rubrics (+ PCT, which loads WITH them) load high on PC1;")
    print("           only MICI + the MITI ratios R:Q/%CR/%MICO stay near-zero = the genuine second axis.")
for arm in sorted(SCO.arm.unique()):
    p = stats.rubric_pca(SCO[SCO.arm == arm], metrics=EXPANDED)
    if p["explained_variance_ratio"]:
        print(f"{arm}: PC1 = {p['explained_variance_ratio'][0]:.1%}  loadings={p['pc1_loadings']}")

### What PC1 measures — factor loadings  `[EVAL]`
Each metric's loading on the principal components. The 5 global-eval rubrics all load high on **PC1** (~0.44 each) → they are essentially **one** halo factor ('good warm therapist' as a single judgment) — and **`PCT` loads high on PC1 with them** (ρ≈0.79–0.94), so despite the design intent it is *not* orthogonal. Only the objective technique ratios **`R:Q` / `%CR` / `%MICO`** and **`MICI ↓`** load ~0 on PC1 and define **PC2** — they are the axis that measures something the halo doesn't.

In [ ]:
fig = plots.factor_loadings_bars(SCO, metrics=EXPANDED, components=("PC1","PC2"))
if fig is not None:
    eda_analysis.save_fig(fig, "factor_loadings", caption="Each metric's PCA loading: the 5 global-eval (halo) rubrics load high on PC1 (one shared factor) and PCT loads high WITH them (~0.79-0.94, not orthogonal as intended); only the MITI ratios R:Q/%CR/%MICO and MICI load ~0 on PC1 and define PC2."); plt.show()
else:
    print("factor loadings need >=2 metrics with enough rows.")

## 2 · Reward-hacking — the headline concern  `[EVAL]`
The synthesis: the warmth **reward proxy** climbs, but the signals *outside* the reward expose the cost.

**Two confounds this section is built to survive.** (i) **Reward = outcome:** Q1+Q2 is *both* the training reward *and* a headline eval metric — so "Q1+Q2 went up" is partly circular and can't, by itself, prove skill gain. (ii) **Shared oracle:** the simulated patient *and* the grader are the **same** model (gpt-4o-mini-2024-07-18), which couples the generator and the evaluator and can inflate the patient-perspective ratings. The reward-hack conclusion therefore does **not** rest on Q1+Q2. It rests on signals that break these confounds, strongest first:
- **Deterministic text metrics** (turn length ↑, loop %, question rate ↓; `behavior.py` regex — see §3 session shape) — computed from the transcript with **no oracle at all**, so they break **both** confounds. These are the load-bearing evidence.
- **Oracle-scored but *un-rewarded* axes** (`MICI ↑`, `PCT` ~flat, the MITI ratios `R:Q`/`%CR`/`%MICO`) — not part of the reward, so they break the reward=outcome circularity; they still share the oracle *model*, so they corroborate rather than prove.

- **2a** — one twin-axis frame: warmth ↑ **and** MI-inconsistency ↑ *together* while patient change-talk stays ~flat → "all rubrics up" is **not** multi-skill.
- **2b** — question-rate cross-check: regex literal-`?` rate vs the oracle question-*function* count (both /turn, same denominator). They agree on **direction** (questions fall) but **diverge in magnitude** — the regex collapses ~7× for GRPO while the oracle drops ~1.6×, because late affirmation/advice turns carry question-function without a literal `?`. Syntax vs function, **not a unit bug** (merge is conv-aligned 96/96).
- **2c** — over-praise cross-check: the lexical marker rate agrees directionally with the oracle's `MICI_OverPraiseRate`.

**The per-questionnaire decompositions behind this synthesis live in `2_Questionnaire_Detail`:** MITI behaviour drift (B6_AF ↑, B3_Q ↓), MICI per-behaviour (the rise is ~all over-praise), PCT patient detail (globals + talk proportions), and the Q2 item-level reward composition (self-disclosure items top the Δ). (The annotated per-metric curves live in `1_Outcomes/trajectories/`; the persona split of the regression in `4_Heterogeneity`.)

In [ ]:
# 2a · The reward-hack in ONE frame — Q1+Q2 reward proxy (left, 1-5) vs MICI/PCT (right, 0-1), twin-axis per arm.
fig = plots.reward_hack_panel(S.SCORES, arms=FOCUS, palette=S.PALETTE)
if fig is not None:
    eda_analysis.save_fig(fig, "reward_hack_panel", caption="Twin-axis per arm: the global-eval reward proxy Q1+Q2 (left, 1-5) rises while MI-Inconsistency (right, higher=worse) rises with it and Patient Change-Talk (right, the actual MI goal) stays ~flat — 'all rubrics up' is not multi-skill.")
    eda_analysis.save_fig(fig, "reward_hack_panel", group="0_headline", caption="The reward-hack in one frame — headline copy; canonical: 3_validity/.")
    plt.show()
else:
    print("no focus arms present.")

In [ ]:
# 2b · Question-rate cross-check — deterministic ?-count/turn vs oracle MITI B3_Q/turn (same denominator).
# NOT a unit bug: both are questions-per-therapist-turn, but the numerators are different constructs —
# literal '?' SYNTAX vs oracle question-FUNCTION. They agree on direction (questions fall) but diverge in
# magnitude for GRPO (regex collapses ~7x, oracle ~1.6x): late affirmation/advice turns carry
# question-function the oracle credits without a literal '?'. The widening gap IS the drift signature.
QRC = behavior.question_rate_crosscheck(S.ARMS)
fig = plots.question_rate_crosscheck(QRC)
if fig is not None:
    eda_analysis.save_fig(fig, "question_rate_crosscheck", caption="Questions per therapist turn two ways: regex literal '?' vs oracle MITI question-function (B3_Q/turn, same denominator). Both fall, but for GRPO the regex rate collapses ~7x while the oracle count drops only ~1.6x — the widening gap = the affirmation/advice drift (declarative prompts carry question-function but no literal '?'). Syntax vs function, not a unit error."); plt.show()
else:
    print("MITI not scored yet — question-rate cross-check empty.")

In [ ]:
# 2c · Over-praise cross-check: deterministic lexical marker rate vs the oracle's MICI_OverPraiseRate.
# Validates the DIRECTION of the demoted regex against the professional coder (same role loop% plays
# for degeneration). Empty until the MICI questionnaire is scored via Run_Eval.
XC = behavior.overpraise_crosscheck(S.ARMS)
if XC.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT','MICI'] to populate this cross-check.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(XC, x="lex_overpraise_marker_rate", y="MICI_OverPraiseRate",
                    hue="arm", palette=figures.arm_palette(sorted(XC.arm.unique())), s=60, ax=ax)
    rho = XC[["lex_overpraise_marker_rate", "MICI_OverPraiseRate"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"Over-praise: lexical marker vs oracle (Spearman rho={rho:.2f})")
    ax.set_xlabel("lexical over-praise marker rate (regex)"); ax.set_ylabel("oracle MICI over-praise rate")
    figures.relabel_legend(ax)
    fig.tight_layout()
    eda_analysis.save_fig(fig, "overpraise_crosscheck", caption="Per-(arm,iteration) deterministic lexical over-praise marker rate vs the oracle-coded MICI over-praise rate — a directional sanity-check on the demoted regex."); plt.show()

## 3 · Session shape — deterministic text metrics  `[EVAL]`
**Purpose.** The no-oracle view of the drift: mean therapist-turn length, regex `?`/turn, degeneration
(loop %) and conversation length across iterations, plus how sessions terminate. These are the
**load-bearing** reward-hack evidence (§2) — computed from the transcripts alone, immune to both the
reward=outcome and shared-oracle confounds. **Read:** rising turn length + falling `?`/turn tracks the
advice/affirmation drift; the end-reason mix is a degeneration health-check. Exported:
`session_shape.png` + tables `session_shape_by_iter` / `session_end_reasons`.

In [ ]:
# 3 · Session shape (deterministic; no oracle) — grid + by-iter table + end-reason mix.
SHAPE = behavior.session_shape_by_iter(S.ARMS)
fig = plots.behavior_trajectory_grid(
    SHAPE, palette=S.PALETTE, metrics=["mean_turn_len", "q_per_turn", "loop", "conv_len"],
    ncols=2, title="Session shape — deterministic text metrics (no oracle)")
if fig:
    eda_analysis.save_fig(fig, "session_shape", caption="Deterministic text metrics per (arm, iteration): mean therapist-turn length, regex ?/turn, degeneration fraction (loop), conversation length. No oracle involved — the load-bearing reward-hack evidence."); plt.show()
ST = SHAPE.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
display(ST)
eda_analysis.save_table(ST, "session_shape_by_iter", caption="Mean deterministic text metrics per (arm, iteration): turn length, ?/turn, loop fraction, conversation length, therapist turns.")

# End-reason mix (who/what terminated each session) — degeneration health-check.
rows = []
for a in S.ARMS:
    for k in a.iters:
        cdir = a.conv_dir(k)
        for fn in (os.listdir(cdir) if cdir and os.path.isdir(cdir) else []):
            if fn.startswith("conversation_") and fn.endswith(".csv"):
                try: dd = pd.read_csv(os.path.join(cdir, fn))
                except Exception: continue
                rows.append({"arm": a.label, "ended_by": str(dd["session_ended_by"].iloc[0]) if "session_ended_by" in dd else "NA"})
if rows:
    END = (pd.DataFrame(rows).groupby(["arm", "ended_by"]).size().rename("n").reset_index()
           .pivot_table(index="arm", columns="ended_by", values="n", fill_value=0).reset_index())
    END.columns.name = None
    display(END)
    eda_analysis.save_table(END, "session_end_reasons", caption="How sessions terminate per arm (count of conversations by session_ended_by) — degeneration health-check.")

## 4 · Persona-matched transcripts  `[EVAL]`
**Purpose.** The same patient persona's conversation early / mid / late per arm — the qualitative drift
in actual words (true-persona recovery makes the match exact across the per-iteration shuffle). Print-only.

In [ ]:
from eda_analysis import persona_order
def file_of_persona(seed, k, pid, n=96): return persona_order(seed, k, n).index(pid)
PERSONA = 0
print("persona", PERSONA, "=", eda_analysis.canonical_personas().loc[PERSONA].to_dict(), "\n")
for arm in S.ARMS:
    iters = arm.iters
    pick = sorted(set([iters[0], iters[len(iters) // 2], iters[-1]]))
    print(f"\n############  {arm.label}  ############")
    for k in pick:
        cdir = arm.conv_dir(k)
        if not cdir: continue
        fi = file_of_persona(arm.seed, k, PERSONA)
        fp = os.path.join(cdir, f"conversation_{fi}.csv")
        if not os.path.exists(fp): continue
        d = pd.read_csv(fp); th = d[d.role == "therapist"]["conversation"].astype(str).tolist()
        print(f"==== {arm.label} model_iter_{k} (conv_{fi}) — {len(th)} therapist turns ====")
        for t in th[1:4]:
            print("   •", " ".join(t.split())[:240]); print()

## 5 · How to read this notebook
- **Factor structure** (§1): the 5 global-eval rubrics alone share a dominant **PC1** (≈91%) — the single-oracle halo; adding the orthogonal axes drops PC1 to ≈55%, but **PCT loads WITH the halo** (ρ≈0.79–0.94, not orthogonal as intended) — the genuine second factor is **`R:Q` / `%CR` / `%MICO` + `MICI ↓`** (objective technique + MI-inconsistency).
- **§2 Reward-hacking** is the synthesis, built to survive the reward=outcome and shared-oracle confounds: the load-bearing evidence is the **deterministic** text metrics (§3, no oracle), corroborated by the un-rewarded oracle axes (MICI ↑, PCT flat).
- **§3 session shape** is those deterministic metrics exported: turn length ↑ + `?`/turn ↓ = the drift; loop % + end-reason mix = degeneration health.
- **Transcripts** (§4) are the ground truth.
- The per-questionnaire decompositions (MITI behaviour drift + thresholds, MICI per-behaviour ≈ all over-praise, PCT patient detail, Q2 item-level reward composition) live in **`2_Questionnaire_Detail`**.
- _(Outcome curves → `1_Outcomes`; persona splits → `4_Heterogeneity`; reward faithfulness → `5_Training_and_Reliability`; exact stats → `7_Stats`.)_

In [ ]:
print("index ->", eda_analysis.build_index())